# SPCA comparison: Bair vs manifold optimization (proper train/test split)

This notebook compares two supervised PCA approaches on the Parkinsons telemonitoring dataset:
- **Bair's method** (feature selection + PCA + regression),
- **Manifold optimization SPCA** (Grassmann optimization with prediction + reconstruction loss).

We use a proper evaluation protocol:
1. Outer 80/20 train/test split (before any preprocessing).
2. Hyperparameters θ (Bair) and λ (manifold) are selected via an inner 80/20 split on the training set.
3. Models are refit on the full training set with the selected HPs.
4. Final metrics are reported on the held-out test set.

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd
import torch

from utils import load_parkinsons_data, compare_bair_vs_manifold

In [2]:
k = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_raw, y_raw, feature_names = load_parkinsons_data(device)

res = compare_bair_vs_manifold(X_raw, y_raw, k=k)

# Unpack for downstream use (e.g. loadings)
L_bair = res['bair']['L']
L_manifold = res['manifold']['L']

print(f"Bair:    theta={res['bair']['theta']:.3f},  "
      f"VE={res['bair']['ve']:.4f},  "
      f"MSE={res['bair']['pred_err']:.4f},  "
      f"runtime={res['bair']['runtime']:.4f}s")
print(f"Manifold: lam={res['manifold']['lam']:.6f},  "
      f"VE={res['manifold']['ve']:.4f},  "
      f"MSE={res['manifold']['pred_err']:.4f},  "
      f"runtime={res['manifold']['runtime']:.4f}s")

/home/jasa/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


theta = 0.100:  val_mse = 0.3445
theta = 0.358:  val_mse = 0.3445
theta = 0.616:  val_mse = 0.3445
theta = 0.874:  val_mse = 0.3445
theta = 1.132:  val_mse = 0.3435
theta = 1.389:  val_mse = 0.3398
theta = 1.647:  val_mse = 0.3381
theta = 1.905:  val_mse = 0.3381
theta = 2.163:  val_mse = 0.3426
theta = 2.421:  val_mse = 0.3426
theta = 2.679:  val_mse = 0.3426
theta = 2.937:  val_mse = 0.3426
theta = 3.195:  val_mse = 0.3426
theta = 3.453:  val_mse = 0.3426
theta = 3.711:  val_mse = 0.3426
theta = 3.968:  val_mse = 0.3422
theta = 4.226:  val_mse = 0.2812
theta = 4.484:  val_mse = 0.2812
theta = 4.742:  val_mse = 0.2949
theta = 5.000:  val_mse = 0.2949

Best theta = 4.226 (val_mse = 0.2812)
Step 0: loss = 181.1735
Step 10: loss = 142.0961
Step 20: loss = 139.7810
Step 30: loss = 138.6750
Step 40: loss = 138.0286
Step 50: loss = 137.6235
Step 60: loss = 137.3373
Step 70: loss = 137.0690
Step 80: loss = 136.6203
Step 90: loss = 136.1260
Step 100: loss = 136.0075
Step 110: loss = 135.9072


## Results summary (held-out test set)

The manifold SPCA typically achieves both higher variance explained and lower prediction error on the test set, at the cost of longer training time.


In [3]:
# Build results table
results = pd.DataFrame({
    "Method": ["Bair SPCA", "Manifold SPCA"],
    "Dataset": ["Parkinsons", "Parkinsons"],
    "Variance explained": [res['bair']['ve'], res['manifold']['ve']],
    "Prediction error (MSE)": [res['bair']['pred_err'], res['manifold']['pred_err']],
    "Runtime (sec)": [res['bair']['runtime'], res['manifold']['runtime']],
})
results.set_index("Method", inplace=True)
results

,Dataset,Variance explained,Prediction error (MSE),Runtime (sec)
Method,,,,
Bair SPCA,Parkinsons,0.048665,0.297345,0.000396
Manifold SPCA,Parkinsons,0.771264,0.234256,1.482067
